In [3]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import os

class ImageCropper:
    def __init__(self, master):
        self.master = master
        master.title("图像裁剪工具 - 专业版")
        master.geometry("1000x800")

        # 初始化变量
        self.image_paths = []
        self.output_dir = ""
        self.crop_coords = None
        self.current_image = None
        self.scale_factor = 1.0

        # 创建界面组件
        self.create_widgets()

    def create_widgets(self):
        # 控制面板框架
        control_frame = tk.Frame(self.master)
        control_frame.pack(side=tk.TOP, pady=10)

        # 按钮
        self.btn_select = tk.Button(control_frame, text="1. 选择图片", command=self.select_images)
        self.btn_dir = tk.Button(control_frame, text="2. 选择输出目录", command=self.select_output_dir)
        self.btn_crop = tk.Button(control_frame, text="3. 选择裁剪区域", command=self.start_crop_selection)
        self.btn_process = tk.Button(control_frame, text="4. 开始批量裁剪", command=self.process_images)
        
        self.btn_select.pack(side=tk.LEFT, padx=5)
        self.btn_dir.pack(side=tk.LEFT, padx=5)
        self.btn_crop.pack(side=tk.LEFT, padx=5)
        self.btn_process.pack(side=tk.LEFT, padx=5)
        
        # 图像显示区域
        self.canvas = tk.Canvas(self.master, cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)

        # 状态栏
        self.status = tk.Label(self.master, text="等待操作...", bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status.pack(side=tk.BOTTOM, fill=tk.X)

        # 绑定鼠标事件
        self.canvas.bind("<ButtonPress-1>", self.start_crop)
        self.canvas.bind("<B1-Motion>", self.update_crop)
        self.canvas.bind("<ButtonRelease-1>", self.end_crop)

    def update_status(self, message):
        self.status.config(text=message)
        self.master.update()

    def select_images(self):
        self.image_paths = filedialog.askopenfilenames(
            title="选择要裁剪的图片",
            filetypes=[("Image files", "*.jpg;*.jpeg;*.png;*.bmp")]
        )
        if self.image_paths:
            self.preview_first_image()
            self.update_status(f"已选择 {len(self.image_paths)} 张图片")

    def select_output_dir(self):
        self.output_dir = filedialog.askdirectory(title="选择输出目录")
        if self.output_dir:
            self.update_status(f"输出目录: {self.output_dir}")

    def preview_first_image(self):
        if self.image_paths:
            try:
                img = Image.open(self.image_paths[0])
                self.show_image(img)
            except Exception as e:
                messagebox.showerror("错误", f"无法打开图片: {str(e)}")

    def show_image(self, img):
        self.current_image = img
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        # 计算缩放比例
        self.scale_factor = min(
            canvas_width / img.width,
            canvas_height / img.height
        )
        new_size = (
            int(img.width * self.scale_factor),
            int(img.height * self.scale_factor)
        )
        
        # 缩放图像
        img_resized = img.resize(new_size, Image.Resampling.LANCZOS)
        self.tk_image = ImageTk.PhotoImage(img_resized)
        
        # 更新画布
        self.canvas.delete("all")
        self.canvas.create_image(
            (canvas_width - new_size[0])//2,
            (canvas_height - new_size[1])//2,
            anchor=tk.NW,
            image=self.tk_image
        )

    def start_crop_selection(self):
        if not self.image_paths:
            messagebox.showwarning("警告", "请先选择图片")
            return
        self.crop_coords = None
        self.update_status("请在图片上拖动鼠标选择裁剪区域")

    def start_crop(self, event):
        if self.current_image:
            self.start_x = self.canvas.canvasx(event.x)
            self.start_y = self.canvas.canvasy(event.y)
            self.rect = self.canvas.create_rectangle(
                self.start_x, self.start_y,
                self.start_x, self.start_y,
                outline="red", width=2
            )

    def update_crop(self, event):
        if self.current_image and hasattr(self, 'rect'):
            cur_x = self.canvas.canvasx(event.x)
            cur_y = self.canvas.canvasy(event.y)
            self.canvas.coords(self.rect, self.start_x, self.start_y, cur_x, cur_y)

    def end_crop(self, event):
        if hasattr(self, 'rect'):
            end_x = self.canvas.canvasx(event.x)
            end_y = self.canvas.canvasy(event.y)
            
            # 转换为原始图像坐标
            img_x0 = int((min(self.start_x, end_x) - self.get_image_position()[0]) / self.scale_factor)
            img_y0 = int((min(self.start_y, end_y) - self.get_image_position()[1]) / self.scale_factor)
            img_x1 = int((max(self.start_x, end_x) - self.get_image_position()[0]) / self.scale_factor)
            img_y1 = int((max(self.start_y, end_y) - self.get_image_position()[1]) / self.scale_factor)
            
            self.crop_coords = (img_x0, img_y0, img_x1, img_y1)
            self.update_status(f"选定区域: {self.crop_coords}")

    def get_image_position(self):
        """获取图像在画布上的显示位置"""
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        img_width = self.current_image.width * self.scale_factor
        img_height = self.current_image.height * self.scale_factor
        return (
            (canvas_width - img_width) // 2,
            (canvas_height - img_height) // 2
        )

    def process_images(self):
        if not self.image_paths:
            messagebox.showwarning("警告", "请先选择图片")
            return
        if not self.output_dir:
            messagebox.showwarning("警告", "请选择输出目录")
            return
        if not self.crop_coords:
            messagebox.showwarning("警告", "请先选择裁剪区域")
            return

        try:
            for idx, path in enumerate(self.image_paths, 1):
                self.update_status(f"正在处理图片 {idx}/{len(self.image_paths)}: {os.path.basename(path)}")
                with Image.open(path) as img:
                    # 检查图像尺寸是否一致
                    if img.size != self.current_image.size:
                        if not messagebox.askyesno("尺寸不一致", 
                            f"图片 {os.path.basename(path)} 的尺寸不同，是否继续？"):
                            continue
                    
                    # 执行裁剪
                    cropped = img.crop(self.crop_coords)
                    
                    # 保存结果
                    output_path = os.path.join(
                        self.output_dir,
                        f"{os.path.basename(path)}"
                    )
                    cropped.save(output_path)
            
            messagebox.showinfo("完成", f"成功处理 {len(self.image_paths)} 张图片")
            self.update_status("就绪")

        except Exception as e:
            messagebox.showerror("错误", f"处理失败: {str(e)}")


if __name__ == "__main__":
    root = tk.Tk()
    app = ImageCropper(root)
    root.mainloop()

In [4]:


"""合并两个表元素"""
import pandas as pd

# 读取表一和表二
df1 = pd.read_excel(r"C:\Users\13404\Desktop\标题和推文合集cleared .xlsx")
df2 = pd.read_excel(r"C:\Users\13404\Desktop\标题和推文合集cleared  - 副本.xlsx")
print(df1.columns)
print(df2.columns)
column_name = '标题'
# 按照A列（酒店链接）进行合并
merged_df = pd.merge(df1, df2, on=column_name, how='inner')
# 垂直合并到基准数据框
# merged_df = pd.concat([merged_df, df], ignore_index=True)
# 输出新的表
print(len(merged_df))

merged_df[column_name] = merged_df[column_name].str.strip()  # 去除首尾空格
merged_df[column_name] = merged_df[column_name].replace('', pd.NA)  # 将空字符串替换为NaN
merged_df.drop_duplicates(subset=column_name, keep='first', inplace=True) # 去重重复值
merged_df.dropna(subset=[column_name], inplace=True)
# 将合并后的结果写入新的 Excel 文件
print(len(merged_df))
outputpath = 'merged_table.xlsx'
merged_df.to_excel(outputpath, index=False)


Index(['链接', '标题', '点赞', '作者', '作者链接', '推文', '收藏', '评论', 'Unnamed: 8'], dtype='object')
Index(['标题', '封面地址'], dtype='object')
295
271
